# ![image-alt-text](https://raw.githubusercontent.com/FabricTools/fabric-icons/refs/heads/main/node_modules/%40fabric-msft/svg-icons/dist/svg/notebook_64_item.svg) 
# DAX Query — `INFO.VIEW.MEASURES()` across Semantic Models

##### Automate collecting all measures from every semantic model — in a single workspace or across all workspaces — and save to a Lakehouse delta table.

_This is a Python notebook designed for Microsoft Fabric._

---

### Two approaches covered
| # | Approach | Scope | Key API |
|---|----------|-------|---------|
| 1 | **Single Workspace** | All semantic models in one workspace | `fabric.list_datasets(workspace=...)` |
| 2 | **All Workspaces** | Every semantic model the user has access to | `fabric.list_workspaces()` → loop `fabric.list_datasets()` |

### How it works
1. **Discover** semantic models using `sempy.fabric` (SemPy) REST mode  
2. **Execute** `EVALUATE INFO.VIEW.MEASURES()` via `fabric.evaluate_dax()`  
3. **Tag** each result row with workspace name, semantic model name, and timestamp  
4. **Save** consolidated results to a Lakehouse delta table via `sempy_labs.save_as_delta_table()`

## Credit & Resources
- [Michael Kovalsky — Semantic Link Labs](https://github.com/microsoft/semantic-link-labs)
- [SemPy docs — `fabric.list_datasets()`](https://learn.microsoft.com/python/api/semantic-link-sempy/sempy.fabric?view=semantic-link-python)
- [SemPy docs — `fabric.list_workspaces()`](https://learn.microsoft.com/python/api/semantic-link-sempy/sempy.fabric?view=semantic-link-python)
- [SemPy docs — `fabric.evaluate_dax()`](https://learn.microsoft.com/fabric/data-science/read-write-power-bi-python)
- [DAX INFO functions reference](https://learn.microsoft.com/en-us/dax/info-functions-dax)
- [Use notebooks with a semantic model](https://learn.microsoft.com/power-bi/transform-model/service-notebooks)

---
## 🔧 Setup & Configuration

In [ ]:
# Install Semantic Link Labs (includes sempy as dependency)
%pip install -U semantic-link-labs

# Import packages
import pandas as pd
from datetime import datetime, timezone
import sempy.fabric as fabric
import sempy_labs as labs

In [ ]:
# ── Configuration ────────────────────────────────────────────────
# Change these values to match your environment

# Lakehouse where results will be saved as a delta table
LakehouseName = "LakehouseDocumentationINFODAX"

# The DAX query to run against every semantic model
DAX_QUERY = """
EVALUATE
INFO.VIEW.MEASURES()
"""

# Delta table name for the consolidated output
DELTA_TABLE_NAME = "info_view_measures_all_models"

In [ ]:
# ── Helper function ──────────────────────────────────────────────
# Reusable function that runs INFO.VIEW.MEASURES() against one semantic model
# and returns a DataFrame with workspace/model metadata columns appended.

def run_info_measures(semantic_model_name: str, workspace_name: str, timestamp: str) -> pd.DataFrame | None:
    """
    Execute INFO.VIEW.MEASURES() on a single semantic model and return the result
    with metadata columns (workspace, semantic_model, query_timestamp).
    Returns None if the query fails (e.g. the model is not queryable).
    """
    try:
        df = fabric.evaluate_dax(
            dataset=semantic_model_name,
            dax_string=DAX_QUERY,
            workspace=workspace_name,
        )
        # Clean column names: remove brackets and special chars
        # e.g. "[Measure Name]" → "Measure Name"
        df.columns = (
            df.columns
            .str.replace(r"[\[\]]", "", regex=True)   # strip [ and ]
            .str.strip()                                # trim whitespace
            .str.replace(r"\s+", "_", regex=True)       # spaces → underscores
        )
        # Tag every row with origin metadata
        df["workspace"] = workspace_name
        df["semantic_model"] = semantic_model_name
        df["query_timestamp"] = timestamp
        return df
    except Exception as e:
        print(f"  ⚠ Skipped  '{semantic_model_name}' in '{workspace_name}': {e}")
        return None

---
## Approach 1 — Single Workspace
Loop through **all semantic models in one workspace** and collect `INFO.VIEW.MEASURES()`.

**How it works:**
1. `fabric.list_datasets(workspace, mode="rest")` returns a DataFrame of every semantic model in the target workspace. Using `mode="rest"` only requires **read** permission (no XMLA ReadWrite needed).
2. We iterate over each model name and run the DAX query.
3. Results are concatenated into a single DataFrame and saved as a delta table.

In [ ]:
# ── Approach 1 — Configuration ───────────────────────────────────
# Set the target workspace name (the one containing your semantic models)
TargetWorkspace = "Fabric Analyst in A Day"   # ← change this

In [ ]:
# ── Step 1.1 — Discover all semantic models in the workspace ────
# mode="rest" uses the Power BI REST API, which only requires read access.
df_models = fabric.list_datasets(workspace=TargetWorkspace, mode="rest")
print(f"Found {len(df_models)} semantic model(s) in workspace '{TargetWorkspace}':\n")

# List available columns to help debug schema changes
print("Available columns:", list(df_models.columns))

# Display only columns that exist. Fallback to show all if specific columns are missing.
expected_cols = [col for col in ["Dataset Name", "Dataset ID"] if col in df_models.columns]
if expected_cols:
    display(df_models[expected_cols].reset_index(drop=True))
else:
    display(df_models.reset_index(drop=True))


In [ ]:
# ── Step 1.2 — Loop and execute DAX query on every model ────────
timestamp = datetime.now(timezone.utc).isoformat()
frames = []

for _, row in df_models.iterrows():
    model_name = row["Dataset Name"]
    print(f"▶ Querying '{model_name}' ...")
    result = run_info_measures(model_name, TargetWorkspace, timestamp)
    if result is not None and not result.empty:
        frames.append(result)
        print(f"  ✔ {len(result)} measure(s) collected")

# Combine all results
if frames:
    df_single_ws = pd.concat(frames, ignore_index=True)
    print(f"\n{'─'*60}")
    print(f"Total measures collected: {len(df_single_ws)}")
    display(df_single_ws)
else:
    df_single_ws = pd.DataFrame()
    print("No measures were collected.")

In [ ]:
# ── Step 1.3 — Save to Lakehouse delta table ───────────────────
if not df_single_ws.empty:
    labs.save_as_delta_table(
        dataframe=df_single_ws,
        delta_table_name=DELTA_TABLE_NAME + "_single_ws",
        write_mode="overwrite",
        merge_schema=True,
        schema=None,
        lakehouse=LakehouseName,
        workspace=None,          # defaults to notebook workspace
    )
    print(f"✔ Saved {len(df_single_ws)} rows to delta table '{DELTA_TABLE_NAME}_single_ws' in '{LakehouseName}'")
else:
    print("Nothing to save — no measures collected.")

---
## Approach 2 — All Workspaces
Loop through **every workspace** the user has access to, then through **every semantic model** in each workspace.

**How it works:**
1. `fabric.list_workspaces()` returns all workspaces you can access.
2. For each workspace, `fabric.list_datasets(workspace, mode="rest")` discovers the semantic models.
3. For each model, we run `INFO.VIEW.MEASURES()` and tag the results.
4. Everything is concatenated and saved to a single delta table.

> **⚠ Heads-up:** This can take several minutes if you have access to many workspaces. Some workspaces or models may be skipped if they are not backed by a capacity that supports XMLA read (e.g. shared/free workspaces, or models where you lack Build permission).

In [ ]:
# ── Step 2.1 — Discover all workspaces ──────────────────────────
df_workspaces = fabric.list_workspaces()
print(f"You have access to {len(df_workspaces)} workspace(s):\n")
display(df_workspaces[["Name", "Id"]].reset_index(drop=True))

In [ ]:
# ── Step 2.2 — Loop all workspaces → all models → run DAX ──────
timestamp = datetime.now(timezone.utc).isoformat()
frames_all = []
skipped_workspaces = []

for _, ws_row in df_workspaces.iterrows():
    ws_name = ws_row["Name"]
    print(f"\n{'═'*60}")
    print(f"Workspace: {ws_name}")
    print(f"{'═'*60}")

    # Discover semantic models in this workspace
    try:
        df_models_ws = fabric.list_datasets(workspace=ws_name, mode="rest")
    except Exception as e:
        print(f"  ⚠ Could not list models: {e}")
        skipped_workspaces.append(ws_name)
        continue

    if df_models_ws.empty:
        print("  (no semantic models found)")
        continue

    print(f"  Found {len(df_models_ws)} model(s)")

    for _, model_row in df_models_ws.iterrows():
        model_name = model_row["Dataset Name"]
        print(f"  ▶ Querying '{model_name}' ...")
        result = run_info_measures(model_name, ws_name, timestamp)
        if result is not None and not result.empty:
            frames_all.append(result)
            print(f"    ✔ {len(result)} measure(s)")

# Combine all results
if frames_all:
    df_all_ws = pd.concat(frames_all, ignore_index=True)
    print(f"\n{'─'*60}")
    print(f"Total measures collected across all workspaces: {len(df_all_ws)}")
    print(f"Workspaces skipped: {len(skipped_workspaces)}")
    display(df_all_ws)
else:
    df_all_ws = pd.DataFrame()
    print("\nNo measures were collected across any workspace.")

In [ ]:
# ── Step 2.3 — Save to Lakehouse delta table ───────────────────
if not df_all_ws.empty:
    labs.save_as_delta_table(
        dataframe=df_all_ws,
        delta_table_name=DELTA_TABLE_NAME + "_all_ws",
        write_mode="overwrite",
        merge_schema=True,
        schema=None,
        lakehouse=LakehouseName,
        workspace=None,
    )
    print(f"✔ Saved {len(df_all_ws)} rows to delta table '{DELTA_TABLE_NAME}_all_ws' in '{LakehouseName}'")
else:
    print("Nothing to save — no measures collected.")

---
## Appendix — Key Concepts & Tips

### API functions used

| Function | Package | Purpose |
|----------|---------|---------|
| `fabric.list_workspaces()` | `sempy.fabric` | Returns a DataFrame of all workspaces the current user can access |
| `fabric.list_datasets(workspace, mode="rest")` | `sempy.fabric` | Lists every semantic model in a workspace. `mode="rest"` requires only read permissions |
| `fabric.evaluate_dax(dataset, dax_string, workspace)` | `sempy.fabric` | Executes a DAX query against a semantic model and returns the result as a pandas DataFrame |
| `labs.save_as_delta_table(...)` | `sempy_labs` | Writes a pandas DataFrame to a Lakehouse delta table |

### Common gotchas

| Issue | Cause | Fix |
|-------|-------|-----|
| `403 Forbidden` on `evaluate_dax` | You need **Build** permission on the semantic model + XMLA read-only must be enabled on the capacity | Ask the workspace admin to grant Build permission |
| `list_datasets` returns empty | The workspace is a "My Workspace" or has no semantic models | Expected — skip it |
| `evaluate_dax` timeout | Large models or throttled capacity | Increase timeout or run during off-peak |
| `merge_schema` errors on delta save | Column types differ between runs | Set `merge_schema=True` (already set above) |

### Switching to `append` mode
If you want to **track measures over time** (e.g. daily snapshots), change `write_mode="overwrite"` to `write_mode="append"` in the `save_as_delta_table` calls. The `query_timestamp` column lets you filter by run date.